# capitulo3_pseudotime — Pseudotime Trajectory Analysis (Part 3)
## Requires: pbmc_harmony_curated.h5ad from capitulo1_single_cell.R

Run each section **in order**. Read the comments in each cell before changing parameters.

---
## SECTION 24 — Setup
Sets paths and loads libraries. Only change `PIPELINE_DIR` and `DATA_DIR` if your folder structure is different.

In [ ]:
import os

# ── Paths (must match capitulo1_single_cell.R) ────────────────────────────────
PIPELINE_DIR = os.path.expanduser("~/projects2/eleo/ScRNA/metodologia/ScRNASeq-Docker/")
DATA_DIR     = os.path.expanduser("~/projects2/eleo/ScRNA/")
base_dir     = os.path.join(DATA_DIR, "metodologia/resultados")

# Load libraries and functions (equivalent to source() in R)
exec(open(os.path.join(PIPELINE_DIR, "load_libraries_python.py")).read(), globals())
exec(open(os.path.join(PIPELINE_DIR, "ScRNA_Pseudotime_Functions.py")).read(), globals())

dir_pseudotime = os.path.join(base_dir, "09_pseudotime")
os.makedirs(dir_pseudotime, exist_ok=True)

print("✓ SECTION 24 COMPLETE: Setup done")

---
## SECTION 25 — Load Data
Loads the AnnData object (`.h5ad`) exported from Seurat by capitulo1.

**What this section does:**
1. Reads the `.h5ad` file with all cells, genes, and metadata from capitulo1
2. Converts the UMAP and PCA coordinates from R format to scanpy format
3. Prints the available cell type names — **you will need these for Section 26**
4. Plots a UMAP as a visual check that the object loaded correctly

**★ Parameter to set:**
- `ANNOTATION_COL`: the column with your final cell type labels
  - Run `print(adata.obs.columns.tolist())` to see all available columns
  - Common options: `celltype_curated`, `celltype_reference`, `seurat_clusters`
- `N_JOBS`: CPU cores for parallel steps (4–8 on a laptop, up to 32 on server)

In [ ]:
# ┌─ PARAMETERS ───────────────────────────────────────────────────────────────
#   ANNOTATION_COL : ★ column in adata.obs with your final cell type labels
#                    Run: print(adata.obs.columns.tolist()) to see all options
#   N_JOBS         : ★ CPU cores (4–8 laptop, up to 32 server)
# └────────────────────────────────────────────────────────────────────────────
ANNOTATION_COL = "celltype_curated"  # ★ change to your cell type column
N_JOBS         = 4                   # ★ adjust to your machine

dir_objects = os.path.join(base_dir, "objects")

# Load the object from capitulo1
adata = sc.read_h5ad(os.path.join(dir_objects, "pbmc_harmony_curated.h5ad"))

# Convert R-format keys to scanpy format
adata.obsm["X_umap"] = adata.obsm["UMAP"].values
adata.obsm["X_pca"]  = adata.obsm["PCA"].values

sc.settings.figdir = dir_pseudotime
# figsize matches R save_pdf default (w=10, h=8); dpi_save=300 for export
sc.set_figure_params(figsize=(10, 8), dpi=80, dpi_save=300)

# ★ Read these names — use them to set TRAJECTORY_CLUSTERS in Section 26
print("Cell types available in", ANNOTATION_COL, ":")
print(sorted(adata.obs[ANNOTATION_COL].unique().tolist()))

# Matches R: label=TRUE, repel=TRUE, raster=FALSE
# show=True displays inline in Jupyter; save also exports the file
sc.pl.umap(
    adata,
    color              = ANNOTATION_COL,
    legend_loc         = "on data",   # label=TRUE in R
    legend_fontsize    = 9,
    legend_fontoutline = 3,           # repel=TRUE equivalent
    frameon            = False,
    show               = True,
    save               = "_overview.png",
)
print("\n✓ SECTION 25 COMPLETE: Object loaded")

---
## SECTION 26 — Trajectory Inference
Builds a tree-shaped trajectory through selected cell types using Palantir + scFates.

**★ Parameters to set:**
- `TRAJECTORY_CLUSTERS`: cell types to include (use names from Section 25 output)
- `ROOT_CLUSTER`: the progenitor / starting cell type

In [ ]:
# ┌─ PARAMETERS ───────────────────────────────────────────────────────────────
#   TRAJECTORY_CLUSTERS : ★ cell types to include, must exist in ANNOTATION_COL
#   ROOT_CLUSTER        : ★ where pseudotime = 0 (least-differentiated type)
#   NODES               : tree resolution (50–200); more = finer detail, slower
#   SIGMA               : smoothing (0.1–0.5); lower = follows cells more tightly
#   PPT_LAMBDA          : complexity (10–200); higher = simpler tree, fewer branches
#   N_EIGS              : diffusion dimensions to keep (must be < 50)
# └────────────────────────────────────────────────────────────────────────────
TRAJECTORY_CLUSTERS = ["Pavement Cell", "Guard Cell", "Meristemoid"]  # ★ adjust
ROOT_CLUSTER        = "Meristemoid"  # ★ progenitor / starting cell type
NODES               = 150
SIGMA               = 0.2
PPT_LAMBDA          = 60
N_EIGS              = 20  # must be < 50

adata_traj = build_pseudotime_trajectory(
    adata          = adata,
    clusters       = TRAJECTORY_CLUSTERS,
    root_cluster   = ROOT_CLUSTER,
    annotation_col = ANNOTATION_COL,
    nodes          = NODES,
    sigma          = SIGMA,
    ppt_lambda     = PPT_LAMBDA,
    n_eigs         = N_EIGS,
    n_components   = 50,
    n_neighbors    = 50,
    seed           = 3,
)

plot_trajectory_graphs(
    adata          = adata_traj,
    name           = "trajectory",
    output_dir     = os.path.join(dir_pseudotime, "trajectory"),
    annotation_col = ANNOTATION_COL,
)
print("\n✓ SECTION 26 COMPLETE")

---
## SECTION 27 — Dendrogram and Milestones
Identifies branch endpoints (milestones). **Read the milestone names printed below before running Section 28.**

In [ ]:
adata_traj = build_dendrogram(adata_traj)

# ★ Copy these milestone names into MILESTONES_TO_ANALYZE in Section 28
print("Available milestones:")
print(sorted(adata_traj.obs["milestones"].unique().tolist()))

scf.pl.dendrogram(adata_traj, color="milestones", legend_loc="on data", save="_milestones.pdf")
print("\n✓ SECTION 27 COMPLETE — see milestone names above")

---
## SECTION 28 — Milestone Analysis
For each branch endpoint: tests which genes change significantly along that path and fits smooth expression curves.

**★ Parameters to set:**
- `MILESTONES_TO_ANALYZE`: use the names printed in Section 27
- `A_CUT`: lower = more genes (noisier), higher = fewer genes (more selective)

In [ ]:
# ┌─ PARAMETERS ───────────────────────────────────────────────────────────────
#   MILESTONES_TO_ANALYZE : ★ adjust after reading Section 27 output
#   A_CUT                 : association threshold (0–1); start at 0.3
#   P_VAL_CUT             : p-value significance threshold
# └────────────────────────────────────────────────────────────────────────────
MILESTONES_TO_ANALYZE = ["Guard_Cell", "Pavement_1"]  # ★ adjust after Section 27
A_CUT     = 0.3
P_VAL_CUT = 0.001

fitted_objects = {}

for milestone in MILESTONES_TO_ANALYZE:
    fitted_objects[milestone] = run_milestone_analysis(
        adata         = adata_traj,
        milestone     = milestone,
        root_milestone= ROOT_CLUSTER,
        output_dir    = os.path.join(dir_pseudotime, "milestones"),
        n_jobs        = N_JOBS,
        a_cut         = A_CUT,
        p_val_cut     = P_VAL_CUT,
        name_file     = "pseudotime",
    )

print("\n✓ SECTION 28 COMPLETE")

---
## SECTION 29 — Gene Expression Trends
Heatmap of the most dynamically expressed genes per branch + ranked CSV table.

**★ Parameters to set:**
- `HIGHLIGHT_GENES`: known marker genes to highlight (leave `[]` if none)

In [ ]:
# ┌─ PARAMETERS ───────────────────────────────────────────────────────────────
#   HIGHLIGHT_GENES : known markers to highlight in the trends heatmap
#                     ★ Use gene IDs for your organism:
#                       Arabidopsis: TAIR IDs e.g. "AT3G24140"
#                       Human/mouse: gene symbols e.g. "MYC", "SOX2"
#                     ★ Leave as [] if you have no prior markers
# └────────────────────────────────────────────────────────────────────────────
HIGHLIGHT_GENES = []  # ★ e.g. ["AT5G53210", "AT3G06120"]

for milestone, adata_fitted in fitted_objects.items():
    plot_gene_trends(
        adata_fitted    = adata_fitted,
        milestone_name  = milestone,
        output_dir      = os.path.join(dir_pseudotime, "trends"),
        highlight_genes = HIGHLIGHT_GENES,
    )
    genes_by_pseudotime_peak(
        adata          = adata_fitted,
        milestone_name = milestone,
        output_dir     = os.path.join(dir_pseudotime, "tables"),
    )

print("\n✓ SECTION 29 COMPLETE")

---
## SECTION 30 — Module Scores *(optional)*
Projects user-defined gene lists onto the trajectory as per-cell scores.
**Skip this section if you do not have custom gene lists.**

**★ Parameters to set:**
- `MODULE_GENE_FILES`: dict of `{label: path_to_file}` — leave `{}` to skip
- `MODULE_ID_COL`: column name with gene IDs in each file

In [ ]:
# ┌─ PARAMETERS ───────────────────────────────────────────────────────────────
#   MODULE_GENE_FILES : ★ add your gene list files here, or leave as {} to skip
#                       e.g. {"my_module": "/path/to/genes.txt"}
#   MODULE_ID_COL     : column with gene IDs in each file
#                       Arabidopsis: usually "ID" or "TAIR"
#                       Human/mouse: usually "gene_id" or "symbol"
# └────────────────────────────────────────────────────────────────────────────
MODULE_GENE_FILES = {}    # ★ e.g. {"cluster1": "/path/to/cluster1_genes.txt"}
MODULE_ID_COL     = "ID"

if MODULE_GENE_FILES:
    for label, fpath in MODULE_GENE_FILES.items():
        df        = pd.read_csv(fpath, sep="\t")
        gene_list = df[MODULE_ID_COL].dropna().unique().tolist()
        compute_module_score(adata_traj, gene_list, label)

    sc.pl.draw_graph(
        adata_traj,
        color  = [f"{label}_module_score" for label in MODULE_GENE_FILES],
        cmap   = "viridis",
        show   = False,
        save   = "_module_scores.png",
    )
    print("✓ Module score plots saved")
else:
    print("Section 30 skipped — no MODULE_GENE_FILES defined")

print("\n✓ SECTION 30 COMPLETE")